# MD_TP150_D14 — WITH EXPIRY TRACKING
Shows entry/exit dates + which weekly expiry cycle each trade belongs to.

In [ ]:
import duckdb, pandas as pd, numpy as np, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
WORK_DIR = Path(r"C:\Users\pc\Downloads\stock hist data\Option Strategy Backtesting")
LOT, DB_PATH = 50, Path(r"C:\Users\pc\Desktop\ucode\project\storage\duckdb\options_history.duckdb")
CUT_TIME = pd.Timestamp("14:15").time()

In [ ]:
# === LOAD SPOT ===
h1 = pd.read_csv(WORK_DIR / "NIFTY50_ONE_HOUR.csv", parse_dates=["datetime"])
m5 = pd.read_csv(WORK_DIR / "NIFTY50_FIVE_MINUTE.csv", parse_dates=["datetime"])
for d in (h1, m5):
    d["datetime"] = pd.to_datetime(d["datetime"], utc=True).dt.tz_convert("Asia/Kolkata")
    d.sort_values("datetime", inplace=True); d.reset_index(drop=True, inplace=True)
me = m5["datetime"].astype("int64").values
m5_times = m5["datetime"].dt.time.values
b = (h1["close"]-h1["open"]).abs(); g = h1["close"]>h1["open"]; rr = h1["close"]<h1["open"]
trades = []
def to_int64(ts): return np.datetime64(ts,"us").astype("int64")
for i in range(1, len(h1)):
    if not (rr.iloc[i-1] and g.iloc[i]): continue
    if h1["open"].iloc[i] > h1["close"].iloc[i-1] or h1["close"].iloc[i] < h1["open"].iloc[i-1]: continue
    if b.iloc[i] < b.iloc[i-1]*0.5 or h1["datetime"].iloc[i].hour == 9: continue
    lv = h1["high"].iloc[i]; ts = h1["datetime"].iloc[i]
    idx = np.searchsorted(me, to_int64(ts), side="right")
    if idx >= len(m5): continue
    bi = idx
    while bi < len(m5) and m5["close"].iloc[bi] <= lv: bi += 1
    if bi >= len(m5)-1: continue
    ri = bi+1
    while ri < len(m5):
        if m5["low"].iloc[ri] < lv and m5["close"].iloc[ri] > lv and m5_times[ri] < CUT_TIME: break
        ri += 1
    if ri >= len(m5): continue
    ep_ = m5["close"].iloc[ri]
    if ep_ - m5["low"].iloc[ri] <= 0: continue
    he = ep_
    for j in range(ri, len(m5)):
        ca = m5["high"].iloc[j] - m5["low"].iloc[j]
        if m5["high"].iloc[j] > he: he = m5["high"].iloc[j]
        if m5["close"].iloc[j] < he - 55*ca:
            trades.append({"entry_dt": m5["datetime"].iloc[ri], "yr": ts.year, "mo": ts.month})
            break
trades = pd.DataFrame(trades)
trades["ed_naive"] = trades["entry_dt"].dt.tz_localize(None)
trades_pre = trades[trades["ed_naive"] >= pd.Timestamp("2021-06-14")].reset_index(drop=True)
print(f"Spot entries: {len(trades_pre)}")

In [ ]:
# === LOAD OPTION DATA (same-strike cache) ===
con = duckdb.connect(str(DB_PATH))
df_atm = con.execute("""SELECT timestamp,close,strike FROM options_data_clean
    WHERE option_type='CALL' AND expiry_code=1 AND expiry_flag='WEEK' AND atm_distance=0 ORDER BY timestamp""").fetchdf()
con.close()
df_atm["timestamp"] = pd.to_datetime(df_atm["timestamp"], utc=False)
atm_ts = df_atm["timestamp"].values.astype("datetime64[us]")
atm_cl = df_atm["close"].values.astype(float)
atm_st = df_atm["strike"].values.astype(float)

def lookup_atm(ed):
    ts64 = np.datetime64(ed, "us"); i = np.searchsorted(atm_ts, ts64)
    if i >= len(atm_ts): return len(atm_ts)-1, atm_cl[-1], atm_st[-1]
    if i == 0: return 0, atm_cl[0], atm_st[0]
    return (i, atm_cl[i], atm_st[i]) if atm_ts[i] == ts64 else (i-1, atm_cl[i-1], atm_st[i-1])

strike_set = set()
for ed in trades_pre["ed_naive"]: _,_,st = lookup_atm(ed); strike_set.add(int(st))
stk_list = sorted(strike_set)
con2 = duckdb.connect(str(DB_PATH))
df_all = con2.execute(f"""SELECT timestamp,close,strike FROM options_data_clean
    WHERE option_type='CALL' AND expiry_code=1 AND expiry_flag='WEEK' AND strike IN ({','.join(map(str,stk_list))})
    ORDER BY strike,timestamp""").fetchdf()
con2.close()
df_all["timestamp"] = pd.to_datetime(df_all["timestamp"], utc=False)
strike_cache = {}
for stk, grp in df_all.groupby("strike"):
    ts = grp["timestamp"].values.astype("datetime64[us]")
    strike_cache[int(stk)] = {"ts": ts, "cl": grp["close"].values.astype(float)}
print(f"Strikes cached: {len(strike_cache)}")

In [ ]:
# === DETECT WEEKLY EXPIRY DATES PER STRIKE ===
# Each strike's data has gaps at expiry. Detect blocks of consecutive data.
def detect_expiries(ts_array):
    """Return list of expiry dates (last timestamp of each data block)."""
    if len(ts_array) < 2: return []
    dates = pd.Series([pd.Timestamp(t).date() for t in ts_array])
    unique_dates = sorted(dates.unique())
    if len(unique_dates) < 2: return []
    # A gap > 1 trading day = expiry boundary
    expiries = []
    for i in range(len(unique_dates)-1):
        d1, d2 = unique_dates[i], unique_dates[i+1]
        if (d2 - d1).days > 3:
            expiries.append(d1)
    expiries.append(unique_dates[-1])  # last block end
    return expiries

strike_expiry_map = {}
for stk, sd in strike_cache.items():
    strike_expiry_map[stk] = detect_expiries(sd["ts"])

def find_expiry_for_date(strike, dt):
    """Return the expiry date that this trade date falls under."""
    exps = strike_expiry_map.get(strike, [])
    dt_date = pd.Timestamp(dt).date()
    for exp_date in exps:
        if dt_date <= exp_date:
            return exp_date
    return exps[-1] if exps else None

print(f"Strikes with expiry data: {len(strike_expiry_map)}")

In [ ]:
# === BUILD TRADE INFO + RUN STRATEGY ===
trade_infos = []
for idx, row in trades_pre.iterrows():
    ts64 = np.datetime64(row["ed_naive"], "us"); i = np.searchsorted(atm_ts, ts64)
    if i >= len(atm_ts): si = len(atm_ts)-1
    elif i == 0: si = 0
    else: si = i if atm_ts[i] == ts64 else i-1
    st = int(atm_st[si]); sd = strike_cache.get(st)
    if sd is None: trade_infos.append(None); continue
    s_idx = np.searchsorted(sd["ts"], atm_ts[si])
    if s_idx >= len(sd["cl"]): trade_infos.append(None); continue
    trade_infos.append({"strike": st, "ep": float(sd["cl"][s_idx]), "s_idx": int(s_idx), "stk_data": sd,
                        "yr": int(row["yr"]), "mo": int(row["mo"]),
                        "entry_dt_ts": sd["ts"][s_idx]})

# Run strategy
TP, MAXD = 150, 14
records = []
for info in trade_infos:
    if info is None: continue
    sd = info["stk_data"]; s_idx = info["s_idx"]; ep = info["ep"]
    end_ns = sd["ts"][s_idx] + np.timedelta64(int(MAXD * 86400 * 1e6), "us")
    result, exit_idx, ex_dt, ex_pr = None, None, None, None
    for i in range(s_idx+1, min(s_idx+3000, len(sd["cl"]))):
        if sd["ts"][i] > end_ns: result = sd["cl"][i]-ep; ex_dt=sd["ts"][i]; ex_pr=sd["cl"][i]; exit_idx=i; break
        if sd["cl"][i] - ep >= TP: result = sd["cl"][i]-ep; ex_dt=sd["ts"][i]; ex_pr=sd["cl"][i]; exit_idx=i; break
    if result is None: continue
    en_dt = sd["ts"][s_idx]
    en_date = pd.Timestamp(en_dt).date()
    ex_date = pd.Timestamp(ex_dt).date()
    hold_d = (ex_date - en_date).days
    en_exp = find_expiry_for_date(info["strike"], en_dt)
    ex_exp = find_expiry_for_date(info["strike"], ex_dt)
    same_weekly = (en_exp == ex_exp) if en_exp and ex_exp else None
    records.append({"entry": en_dt, "exit": ex_dt, "hold_days": hold_d,
                    "strike": info["strike"], "ep": round(ep,1), "exit_p": round(ex_pr,1),
                    "pnl": round(result,1), "pnl_rs": int(round(result,1)*LOT),
                    "entry_expiry": str(en_exp) if en_exp else "?",
                    "exit_expiry": str(ex_exp) if ex_exp else "?",
                    "same_weekly": same_weekly})

trade_df = pd.DataFrame(records)
trade_df["#"] = range(1, len(trade_df)+1)
print(f"Trades executed: {len(trade_df)}")

In [ ]:
# === PRINT ALL TRADES WITH EXPIRY INFO ===
pd.set_option("display.max_rows", 400)
pd.set_option("display.width", 200)
display_df = trade_df[["#","entry","exit","hold_days","strike","ep","exit_p","pnl","pnl_rs",
                       "entry_expiry","exit_expiry","same_weekly"]].copy()
display_df["entry"] = display_df["entry"].apply(lambda x: pd.Timestamp(x).strftime("%Y-%m-%d"))
display_df["exit"] = display_df["exit"].apply(lambda x: pd.Timestamp(x).strftime("%Y-%m-%d"))
print(f"TRADE BOOK — MD_TP150_D14  ({len(trade_df)} trades)\n")
display(display_df.style.format({"pnl": "{:+.1f}", "pnl_rs": "Rs {:+,.0f}"}).set_table_attributes('style="font-size:9px"'))

In [ ]:
# === EXPIRY CROSS ANALYSIS ===
total = len(trade_df)
same = trade_df["same_weekly"].sum()
cross = total - same
print(f"Total trades: {total}")
print(f"Same weekly cycle: {same} ({same/total*100:.1f}%)")
print(f"Crossed expiry:    {cross} ({cross/total*100:.1f}%)")

# Stats for safe trades (same weekly)
safe = trade_df[trade_df["same_weekly"] == True]
if len(safe) > 0:
    print(f"\nSAME WEEKLY ({len(safe)} trades):")
    print(f"  Net: {safe['pnl'].sum():+,.0f} pts (Rs {safe['pnl_rs'].sum():+,.0f})")
    print(f"  WR: {(safe['pnl']>0).mean()*100:.1f}%  Avg: {safe['pnl'].mean():+.1f}")

crossed = trade_df[trade_df["same_weekly"] == False]
if len(crossed) > 0:
    print(f"\nCROSSED EXPIRY ({len(crossed)} trades) — INVALID PnL:")
    print(f"  Net: {crossed['pnl'].sum():+,.0f} pts (Rs {crossed['pnl_rs'].sum():+,.0f})")
    print(f"  WR: {(crossed['pnl']>0).mean()*100:.1f}%  Avg: {crossed['pnl'].mean():+.1f}")
    print(f"  These compare prices of DIFFERENT option contracts across weekly expiry")

In [ ]:
# === SUMMARY STATS (SAME-WEEKLY ONLY) ===
pnls = safe["pnl"].values if len(safe) > 0 else trade_df["pnl"].values
n = len(pnls); net = float(pnls.sum())
wr = float((pnls>0).mean()*100); avg = float(pnls.mean())
std = float(pnls.std()) if n>1 else 0
sharpe = float(avg/std*np.sqrt(252)) if std>0 else 0
cum = np.cumsum(pnls); mx = np.maximum.accumulate(cum)
mdd = float((mx-cum).max()) if n>0 else 0
calmar = float(net/mdd) if mdd>0 else 0
pf = float(pnls[pnls>0].sum()/abs(pnls[pnls<0].sum())) if (pnls<0).sum()>0 else 999
wl = float(pnls[pnls>0].mean()/abs(pnls[pnls<0].mean())) if (pnls<0).sum()>0 else 999
print(f"{'='*55}")
print(f"MD_TP150_D14 — SAME WEEKLY ONLY (valid results)")
print(f"{'='*55}")
print(f"Trades:          {n}")
print(f"Net PnL:         {net:>+8,.0f} pts  (Rs {net*LOT:>+,.0f})")
print(f"Win Rate:        {wr:.1f}%")
print(f"Avg PnL:         {avg:+.1f} pts")
print(f"Sharpe:          {sharpe:.2f}")
print(f"Calmar:          {calmar:.1f}x")
print(f"Profit Factor:   {pf:.2f}x")
print(f"Max DD:          {mdd:>+8,.0f} pts  (Rs {mdd*LOT:>+,.0f})")

print(f"\n{'='*55}")
print(f"ALL TRADES (including cross-expiry — INVALID)")
print(f"{'='*55}")
pnls_all = trade_df["pnl"].values
n_all = len(pnls_all); net_all = float(pnls_all.sum())
wr_all = float((pnls_all>0).mean()*100); avg_all = float(pnls_all.mean())
print(f"Trades:          {n_all}")
print(f"Net PnL:         {net_all:>+8,.0f} pts  (Rs {net_all*LOT:>+,.0f})")
print(f"Win Rate:        {wr_all:.1f}%")
print(f"Avg PnL:         {avg_all:+.1f} pts")